# SwinUNETR Self-Supervised Pretraining on Unlabeled Data

## What this notebook does

Your situation:
- **Labeled data** (gliomas) → already trained → Dice 0.81
- **Unlabeled data** (meningiomas) → no ground truth → model has never seen them

**Self-supervised learning** solves this by inventing *proxy tasks* that require
no labels but force the encoder to learn meaningful anatomy representations.
After pretraining on the unlabeled meningioma MRIs, you fine-tune the entire
SwinUNETR (encoder + decoder) on your labeled glioma data.

## How it works — 3 proxy tasks (CVPR 2022, Hatamizadeh et al.)

```
Unlabeled MRI volume
        │
        ├─► Mask 75% patches ──► Encoder ──► Inpainting Head ──► reconstruct masked regions
        │                                                           (L1 loss)
        ├─► Rotate 0/90/180/270° ──► Encoder ──► Rotation Head ──► predict rotation class
        │                                                           (CE loss)
        └─► Two augmented views ──► Encoder ──► Contrastive Head ─► pull views together
                                                                     (NT-Xent loss)
```

The encoder learns:
- **Inpainting** → texture, structure, spatial context of brain/tumour anatomy
- **Rotation**   → anatomical orientation and shape understanding
- **Contrastive**→ invariance to intensity/noise while preserving structure

## 2-stage pipeline

```
Stage 1 — SSL Pretraining (this notebook)
  Unlabeled meningioma MRIs → train SwinUNETR encoder with 3 proxy tasks
  → save pretrained encoder weights

Stage 2 — Supervised Fine-tuning (existing training notebook)
  Load pretrained encoder weights into SwinUNETR
  → fine-tune on labeled glioma + any available labeled data
  → expect +3–8% Dice improvement on meningiomas
```

**Label map (for reference):** `0`=BG · `1`=NCR · `2`=ED · `3`=ET

---
**Only edit Cell 2. Run all cells top to bottom.**

## Cell 1 — Imports

In [1]:
import os, time, random, warnings
from pathlib import Path
from datetime import datetime
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader

torch.backends.cudnn.benchmark = True
import torch._dynamo
torch._dynamo.config.suppress_errors = True

from monai.utils import set_determinism
from monai.networks.nets import SwinUNETR
from monai.losses import ContrastiveLoss
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
    NormalizeIntensityd, CropForegroundd, RandSpatialCropd,
    RandFlipd, RandRotate90d, RandGaussianNoised,
    RandScaleIntensityd, RandShiftIntensityd,
)
from monai.data import Dataset as MonaiDataset, DataLoader as MonaiDataLoader, pad_list_data_collate
from monai.config import print_config

NUM_GPUS = torch.cuda.device_count()
DEVICE   = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print_config()
print(f'\nPyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
print(f'GPUs: {NUM_GPUS}')
for i in range(NUM_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  ({p.total_memory/1e9:.1f} GB)')


MONAI version: 1.5.0
Numpy version: 2.0.2
Pytorch version: 2.6.0+cu124
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: d388d1c6fec8cb3a0eebee5b5a0b9776ca59ca83
MONAI __file__: /home/cbme/phd/<username>/.local/lib/python3.9/site-packages/monai/__init__.py

Optional dependencies:
Pytorch Ignite version: 0.4.11
ITK version: 5.4.4
Nibabel version: 5.3.2
scikit-image version: 0.24.0
scipy version: 1.13.1
Pillow version: 11.2.1
Tensorboard version: 2.19.0
gdown version: 5.2.0
TorchVision version: 0.21.0+cu124
tqdm version: 4.66.4
lmdb version: 1.6.2
psutil version: 7.0.0
pandas version: 2.3.0+4.g1dfc98e16a
einops version: 0.8.1
transformers version: 4.40.2
mlflow version: 3.1.0
pynrrd version: 1.1.3
clearml version: NOT INSTALLED or UNKNOWN VERSION.

For details about installing the optional dependencies, please visit:
    https://docs.monai.io/en/latest/installation.html#installing-the-recommended-dependencies


PyTorch 2.6.0+cu124 | CUDA: True
GPUs: 

## Cell 2 — Configuration
**Edit the three paths. Everything else is pre-tuned for 2× V100-32GB.**

In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  EDIT THESE PATHS                                           ║
# ╚══════════════════════════════════════════════════════════════╝

# Folder containing UNLABELED meningioma cases
# Each sub-folder must have *-t1n.nii.gz *-t1c.nii.gz *-t2w.nii.gz *-t2f.nii.gz
# NO seg file needed — this is self-supervised, no labels used
UNLABELED_ROOT = Path(r'/home/cbme/phd/bmz228464/Brats2026/GOAT/Data/Data_without_ground_truth/MICCAI2024-BraTS-GoAT-TrainingData-WithOut-GroundTruth/')


# Where to save the pretrained weights
PRETRAIN_OUT   = Path(r'/home/cbme/phd/bmz228464/Brats2026/GOAT/Model/model_SSL/')

# (Optional) Path to your existing trained SwinUNETR weights
# Set to None to start SSL from scratch, or point to your .pth
# to continue from your already-trained glioma model
INIT_WEIGHTS   = Path(r'/home/cbme/phd/bmz228464/Brats2026/GOAT/Model/Model_New_GT/SwinUNETR_GoAT/run_20260519_2012/best_swinunetr_goat.pth')
# INIT_WEIGHTS = None  # uncomment to start SSL from random init

# ── Architecture — must match your trained model ──────────────────────────────
FEATURE_SIZE   = 48     # 62 M params
IMAGE_KEYS     = ['t1n', 't1c', 't2w', 't2f']
IN_CHANNELS    = 4
OUT_CHANNELS   = 4

# ── SSL Pretraining settings ──────────────────────────────────────────────────
PATCH_SIZE          = (96, 96, 96)   # crop size fed to encoder
MASK_RATIO          = 0.75           # fraction of voxels masked for inpainting
SSL_EPOCHS          = 100            # pretraining epochs (50–200 typical)
SSL_LR              = 1e-4
SSL_WEIGHT_DECAY    = 1e-5
SSL_BATCH_SIZE      = 2 * max(NUM_GPUS, 1)  # 2 per GPU
SSL_NUM_WORKERS     = 8
USE_AMP             = True

# Loss weights for the three proxy tasks
W_INPAINT     = 1.0    # reconstruction weight
W_ROTATION    = 1.0    # rotation prediction weight
W_CONTRAST    = 0.1    # contrastive weight (lower — avoids representation collapse)
TEMPERATURE   = 0.05   # contrastive softmax temperature

# ── Output paths ─────────────────────────────────────────────────────────────
PRETRAIN_OUT.mkdir(parents=True, exist_ok=True)
SSL_ENCODER_PATH   = PRETRAIN_OUT / 'ssl_pretrained_encoder.pth'
SSL_FULL_PATH      = PRETRAIN_OUT / 'ssl_pretrained_full.pth'
SSL_LOG_PATH       = PRETRAIN_OUT / 'ssl_training_log.csv'

SEED = 42
set_determinism(seed=SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print(f'Unlabeled data  : {UNLABELED_ROOT}')
print(f'Output          : {PRETRAIN_OUT}')
print(f'Init weights    : {INIT_WEIGHTS}')
print(f'SSL epochs      : {SSL_EPOCHS}')
print(f'Patch size      : {PATCH_SIZE}')
print(f'Mask ratio      : {MASK_RATIO}')
print(f'Device          : {DEVICE}')


Unlabeled data  : /home/cbme/phd/bmz228464/Brats2026/GOAT/Data/Data_without_ground_truth/MICCAI2024-BraTS-GoAT-TrainingData-WithOut-GroundTruth
Output          : /home/cbme/phd/bmz228464/Brats2026/GOAT/Model/model_SSL
Init weights    : /home/cbme/phd/bmz228464/Brats2026/GOAT/Model/Model_New_GT/SwinUNETR_GoAT/run_20260519_2012/best_swinunetr_goat.pth
SSL epochs      : 100
Patch size      : (96, 96, 96)
Mask ratio      : 0.75
Device          : cuda:0


## Cell 3 — Discover Unlabeled Cases

No seg file required. Any case folder with 4 modalities is included.

In [3]:
import gzip
from tqdm.notebook import tqdm

# ── File integrity checker ────────────────────────────────────────────────────
def check_nifti_gz(path):
    """
    Returns True if the .nii.gz file can be decompressed without errors.
    The zlib.error crash in the DataLoader happens because nibabel tries to
    read a corrupt .nii.gz at runtime inside a worker — crashing the whole run.
    Pre-scanning every file here means corrupt files are skipped before training.
    """
    try:
        with gzip.open(str(path), 'rb') as f:
            f.read(352)   # read NIfTI header bytes only — fast check
        return True
    except Exception:
        return False


def find_modality(case_dir, key):
    hits = sorted(case_dir.glob(f'*-{key}.nii.gz')) or \
           sorted(case_dir.glob(f'*{key}*.nii*'))
    return str(hits[0]) if hits else None


unlabeled_dicts = []
skipped_missing = []
skipped_corrupt = []

case_dirs = sorted(p for p in UNLABELED_ROOT.iterdir() if p.is_dir())
print(f"Scanning {len(case_dirs)} case folders for missing or corrupt files...")

for d in tqdm(case_dirs, desc="Validating files"):
    sid  = d.name
    mods = {k: find_modality(d, k) for k in IMAGE_KEYS}

    # Skip if any modality is missing
    missing = [k for k, v in mods.items() if v is None]
    if missing:
        skipped_missing.append((sid, f'missing: {missing}'))
        continue

    # Skip if any .nii.gz file is corrupt (zlib decompression fails)
    corrupt = [k for k, path in mods.items() if not check_nifti_gz(path)]
    if corrupt:
        skipped_corrupt.append((sid, f'corrupt .nii.gz: {corrupt}'))
        continue

    unlabeled_dicts.append({**mods, 'subject_id': sid})

print(f'\nValid cases ready for SSL  : {len(unlabeled_dicts)}')
print(f'Skipped — missing modality : {len(skipped_missing)}')
print(f'Skipped — corrupt file     : {len(skipped_corrupt)}')

if skipped_corrupt:
    print(f'\nCorrupt files (fix or delete these):')
    for sid, reason in skipped_corrupt:
        print(f'  {sid}: {reason}')
    print()
    print('TIP: Re-download these cases. The .nii.gz files were not fully')
    print('     transferred or were truncated during extraction.')

if not unlabeled_dicts:
    raise RuntimeError(
        f'No valid unlabeled cases found in {UNLABELED_ROOT}.\n'
        'Check that the dataset was fully downloaded and extracted.'
    )

# Show one valid example
ex = unlabeled_dicts[0]
print(f'\nFirst valid case: {ex["subject_id"]}')
for k in IMAGE_KEYS:
    print(f'  {k}: {ex[k]}')


Scanning 1138 case folders for missing or corrupt files...


Validating files:   0%|          | 0/1138 [00:00<?, ?it/s]


Valid cases ready for SSL  : 1138
Skipped — missing modality : 0
Skipped — corrupt file     : 0

First valid case: BraTS-GoAT-00001
  t1n: /home/cbme/phd/bmz228464/Brats2026/GOAT/Data/Data_without_ground_truth/MICCAI2024-BraTS-GoAT-TrainingData-WithOut-GroundTruth/BraTS-GoAT-00001/BraTS-GoAT-00001-t1n.nii.gz
  t1c: /home/cbme/phd/bmz228464/Brats2026/GOAT/Data/Data_without_ground_truth/MICCAI2024-BraTS-GoAT-TrainingData-WithOut-GroundTruth/BraTS-GoAT-00001/BraTS-GoAT-00001-t1c.nii.gz
  t2w: /home/cbme/phd/bmz228464/Brats2026/GOAT/Data/Data_without_ground_truth/MICCAI2024-BraTS-GoAT-TrainingData-WithOut-GroundTruth/BraTS-GoAT-00001/BraTS-GoAT-00001-t2w.nii.gz
  t2f: /home/cbme/phd/bmz228464/Brats2026/GOAT/Data/Data_without_ground_truth/MICCAI2024-BraTS-GoAT-TrainingData-WithOut-GroundTruth/BraTS-GoAT-00001/BraTS-GoAT-00001-t2f.nii.gz


## Cell 4 — SSL Dataset

Each `__getitem__` returns **three tensors** from the same volume:

| Key | What it is | Used for |
|-----|-----------|----------|
| `x_masked` | Volume with 75% of voxels zeroed out | Inpainting proxy task |
| `x_orig` | Clean normalised volume | Inpainting target + contrastive view 1 |
| `x_aug` | Augmented version (noise/scale/flip) | Contrastive view 2 |
| `x_rot` | Rotated volume | Rotation proxy task input |
| `rot_label` | Rotation class (0/1/2/3) | Rotation proxy task target |
| `mask` | Boolean tensor (True = masked) | Inpainting loss mask |

In [4]:
import nibabel as nib
import numpy as np

# ── Preprocessing transforms ──────────────────────────────────────────────────
preprocess_tf = Compose([
    LoadImaged(keys=IMAGE_KEYS),
    EnsureChannelFirstd(keys=IMAGE_KEYS),
    Orientationd(keys=IMAGE_KEYS, axcodes='RAS'),
    NormalizeIntensityd(keys=IMAGE_KEYS, nonzero=True, channel_wise=True),
    CropForegroundd(keys=IMAGE_KEYS, source_key='t1c', allow_smaller=True),
    RandSpatialCropd(
        keys=IMAGE_KEYS,
        roi_size=PATCH_SIZE,
        random_size=False,
    ),
])

# ── Augmentation for the contrastive view ────────────────────────────────────
augment_tf = Compose([
    RandFlipd(keys=IMAGE_KEYS, spatial_axis=[0], prob=0.5),
    RandFlipd(keys=IMAGE_KEYS, spatial_axis=[1], prob=0.5),
    RandFlipd(keys=IMAGE_KEYS, spatial_axis=[2], prob=0.5),
    RandRotate90d(keys=IMAGE_KEYS, prob=0.5, max_k=3),
    RandScaleIntensityd(keys=IMAGE_KEYS, factors=0.15, prob=0.7),
    RandShiftIntensityd(keys=IMAGE_KEYS, offsets=0.15, prob=0.7),
    RandGaussianNoised(keys=IMAGE_KEYS, std=0.03, prob=0.5),
])


# ── Helper: strip MetaTensor metadata completely ──────────────────────────────
# MONAI's MetaTensor carries spatial metadata (affine, applied_operations, etc.)
# that causes torch's default_collate to fail with shape mismatches when
# samples have slightly different raw sizes (before CropForeground).
# Converting to plain numpy and back to a vanilla torch.Tensor removes all
# metadata so the DataLoader can batch them without errors.

def _meta_to_plain(sample_dict, keys):
    """
    Stack MetaTensor modalities into a single plain torch.Tensor.
    Each modality is (1, D, H, W) after EnsureChannelFirstd.
    Output: (C, D, H, W) plain float32 torch.Tensor — no MetaTensor metadata.
    """
    arrays = [np.asarray(sample_dict[k]).squeeze(0) for k in keys]
    stacked = np.stack(arrays, axis=0)          # (C, D, H, W) numpy
    return torch.from_numpy(stacked.copy()).float()   # plain Tensor


# ── SSL Dataset ───────────────────────────────────────────────────────────────

class SSLUnlabeledDataset(Dataset):
    """
    Returns 5 tensors per sample — all plain torch.Tensors, no MetaTensor.
    No ground-truth label is used anywhere.

    Proxy task inputs:
      x_orig    : clean volume (4, D, H, W)
      x_masked  : 75% voxels zeroed  → Inpainting target
      x_aug     : augmented view     → Contrastive view 2
      x_rot     : rotated volume     → Rotation prediction input
      rot_label : int (0-3)          → Rotation prediction target
      mask      : bool (D, H, W)     → which voxels were masked
    """

    # Rotation axes for 4D (C,D,H,W) tensors
    ROTATION_AXES = [
        (None, None),   # class 0 — no rotation
        (1, 2),         # class 1 — rotate D-H plane
        (1, 3),         # class 2 — rotate D-W plane
        (2, 3),         # class 3 — rotate H-W plane
    ]

    def __init__(self, data_dicts, mask_ratio=0.75):
        self.data       = data_dicts
        self.mask_ratio = mask_ratio
        self.monai_ds   = MonaiDataset(data=data_dicts, transform=preprocess_tf)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Try loading the case — if file is corrupt, return the NEXT valid case
        # This is a belt-and-suspenders guard after the pre-scan in Cell 3.
        try:
            sample = self.monai_ds[idx]
        except Exception as e:
            import warnings
            warnings.warn(f"Skipping corrupt case {self.data[idx]['subject_id']}: {e}")
            return self.__getitem__((idx + 1) % len(self))

        # ── 1. Convert MetaTensors → plain (C,D,H,W) float Tensor ────────────
        x_orig = _meta_to_plain(sample, IMAGE_KEYS)   # (4, 96, 96, 96) Tensor

        # ── 2. Masked volume for inpainting ──────────────────────────────────
        # Build a (D,H,W) spatial mask, then expand to all channels
        spatial_mask = torch.rand(*x_orig.shape[1:]) < self.mask_ratio
        x_masked     = x_orig.clone()
        x_masked[spatial_mask.unsqueeze(0).expand_as(x_orig)] = 0.0

        # ── 3. Augmented contrastive view ─────────────────────────────────────
        aug_dict = {k: sample[k] for k in IMAGE_KEYS}
        aug_dict = augment_tf(aug_dict)
        x_aug    = _meta_to_plain(aug_dict, IMAGE_KEYS)   # plain Tensor

        # ── 4. Rotated volume for rotation prediction ─────────────────────────
        rot_class = random.randint(0, 3)
        axes      = self.ROTATION_AXES[rot_class]
        if axes[0] is None:
            x_rot = x_orig.clone()
        else:
            x_rot = torch.rot90(x_orig, k=1, dims=list(axes))

        return {
            'x_orig'    : x_orig,         # (4, D, H, W) plain Tensor
            'x_masked'  : x_masked,       # (4, D, H, W) plain Tensor
            'x_aug'     : x_aug,          # (4, D, H, W) plain Tensor
            'x_rot'     : x_rot,          # (4, D, H, W) plain Tensor
            'rot_label' : torch.tensor(rot_class, dtype=torch.long),
            'mask'      : spatial_mask,   # (D, H, W) bool
            'subject_id': self.data[idx]['subject_id'],
        }


ssl_ds = SSLUnlabeledDataset(unlabeled_dicts, mask_ratio=MASK_RATIO)

ssl_loader = DataLoader(
    ssl_ds,
    batch_size         = SSL_BATCH_SIZE,
    shuffle            = True,
    num_workers        = SSL_NUM_WORKERS,
    pin_memory         = torch.cuda.is_available(),
    persistent_workers = SSL_NUM_WORKERS > 0,
    drop_last          = True,
)

# Sanity check on one batch
sample_b = next(iter(ssl_loader))
print(f'SSL DataLoader ready — {len(ssl_loader)} batches per epoch')
print(f"  x_orig  : {sample_b['x_orig'].shape}   type={type(sample_b['x_orig']).__name__}")
print(f"  x_masked: {sample_b['x_masked'].shape}")
print(f"  x_aug   : {sample_b['x_aug'].shape}")
print(f"  x_rot   : {sample_b['x_rot'].shape}")
print(f"  rot_lbl : {sample_b['rot_label'].tolist()}")
print(f"  mask frac: {sample_b['mask'].float().mean().item():.3f}  (target {MASK_RATIO})")


SSL DataLoader ready — 284 batches per epoch
  x_orig  : torch.Size([4, 4, 96, 96, 96])   type=Tensor
  x_masked: torch.Size([4, 4, 96, 96, 96])
  x_aug   : torch.Size([4, 4, 96, 96, 96])
  x_rot   : torch.Size([4, 4, 96, 96, 96])
  rot_lbl : [1, 0, 3, 1]
  mask frac: 0.750  (target 0.75)


## Cell 5 — Build SwinUNETR with SSL Heads

Three lightweight heads are attached to the SwinUNETR encoder **only for pretraining**.
They are discarded after pretraining — only the encoder weights are saved and loaded
into the fine-tuning model.

```
SwinUNETR encoder (swinViT)          ← pre-trained
    │
    ├── Inpainting Head  ──► (B, 4, D, H, W) reconstruction
    ├── Rotation Head    ──► (B, 4) class logits
    └── Projection Head  ──► (B, 128) unit-sphere embedding

SwinUNETR decoder (encoder1..4, decoder1..4) ← NOT used in pretraining
```

In [5]:
class SwinUNETR_SSL(nn.Module):
    """
    SwinUNETR encoder + three self-supervised proxy heads.
    The decoder is NOT used during pretraining.

    After pretraining, call .save_encoder_weights(path) to extract
    only the encoder parameters for loading into the fine-tuning model.
    """

    def __init__(self, feature_size=48):
        super().__init__()

        # Full SwinUNETR — encoder + decoder
        # We only USE the encoder (swinViT) during SSL; the decoder is loaded
        # fresh during fine-tuning
        self.backbone = SwinUNETR(
            in_channels       = IN_CHANNELS,
            out_channels      = OUT_CHANNELS,
            feature_size      = feature_size,
            use_checkpoint    = True,
            spatial_dims      = 3,
        )

        # Bottleneck dimension: feature_size × 16
        # (48 × 16 = 768 for the full model)
        enc_dim = feature_size * 16

        # ── Head 1: Masked Volume Inpainting ──────────────────────────────────
        # Takes bottleneck features → reconstructs full input volume
        # GroupNorm instead of InstanceNorm (works when spatial size = 1×1×1)
        self.inpaint_head = nn.Sequential(
            nn.Conv3d(enc_dim, enc_dim // 2, kernel_size=1),
            nn.GroupNorm(8, enc_dim // 2),
            nn.GELU(),
            nn.Conv3d(enc_dim // 2, IN_CHANNELS, kernel_size=1),
        )

        # ── Head 2: Rotation Prediction ───────────────────────────────────────
        # Global average pool → 4-class classifier
        self.rot_head = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Linear(enc_dim, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 4),
        )

        # ── Head 3: Contrastive Projection ────────────────────────────────────
        # Projects to 128-dim unit sphere for NT-Xent contrastive loss
        self.proj_head = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Linear(enc_dim, 256),
            nn.GELU(),
            nn.Linear(256, 128),
        )

    def _encode(self, x):
        """
        Run the SwinTransformer encoder and return the bottleneck feature map.
        Returns: (B, enc_dim, d, h, w)
        """
        hidden = self.backbone.swinViT(x, self.backbone.normalize)
        return hidden[4]   # deepest encoder level = bottleneck

    def forward_inpaint(self, x_masked, x_orig):
        """
        Reconstruct the full volume from the masked input.
        Returns reconstructed volume same shape as x_orig.
        """
        feat  = self._encode(x_masked)                     # (B, enc_dim, d, h, w)
        recon = self.inpaint_head(feat)                    # (B, 4, d, h, w)
        recon = F.interpolate(
            recon, size=x_orig.shape[2:],
            mode='trilinear', align_corners=False
        )                                                   # (B, 4, D, H, W)
        return recon

    def forward_rotation(self, x_rot):
        """Predict which 90° rotation was applied. Returns (B, 4) logits."""
        feat = self._encode(x_rot)
        return self.rot_head(feat)

    def forward_contrastive(self, x):
        """Project to unit-sphere embedding. Returns (B, 128)."""
        feat = self._encode(x)
        z    = self.proj_head(feat)
        return F.normalize(z, dim=1)

    def save_encoder_weights(self, path):
        """
        Save ONLY the SwinTransformer encoder weights.
        These are loaded into the fine-tuning SwinUNETR to initialise
        the encoder with SSL-learned representations.
        """
        encoder_state = self.backbone.swinViT.state_dict()
        torch.save(encoder_state, str(path))
        print(f'Encoder weights saved → {path}')

    def save_full_weights(self, path):
        """Save the entire SwinUNETR backbone (encoder + decoder)."""
        torch.save(self.backbone.state_dict(), str(path))
        print(f'Full backbone saved → {path}')


# ── Build and optionally initialise from existing glioma weights ──────────────
ssl_model = SwinUNETR_SSL(feature_size=FEATURE_SIZE).to(DEVICE)

if INIT_WEIGHTS is not None and Path(str(INIT_WEIGHTS)).exists():
    # Load previously trained glioma weights as starting point
    # This is BETTER than random init — you start SSL from a model that
    # already understands brain anatomy
    ckpt = torch.load(str(INIT_WEIGHTS), map_location='cpu')
    # Strip any DataParallel prefixes
    ckpt = {k.replace('module.','').replace('_orig_mod.',''): v
            for k, v in ckpt.items()}
    missing, unexpected = ssl_model.backbone.load_state_dict(ckpt, strict=False)
    print(f'Loaded glioma weights. Missing: {len(missing)}  Unexpected: {len(unexpected)}')
    print('SSL pretraining will refine these weights on meningioma anatomy.')
else:
    print('Starting SSL from random initialisation (no init weights found).')

# DataParallel
if NUM_GPUS > 1:
    ssl_model = nn.DataParallel(ssl_model)
    print(f'DataParallel across {NUM_GPUS} GPUs')

n = sum(p.numel() for p in ssl_model.parameters())
print(f'SSL model parameters: {n/1e6:.2f} M')

# ── Losses ────────────────────────────────────────────────────────────────────
contrastive_loss_fn = ContrastiveLoss(temperature=TEMPERATURE)

# ── Optimiser ─────────────────────────────────────────────────────────────────
try:
    ssl_optimizer = torch.optim.AdamW(
        ssl_model.parameters(),
        lr=SSL_LR, weight_decay=SSL_WEIGHT_DECAY, fused=True,
    )
    print('Fused AdamW applied')
except TypeError:
    ssl_optimizer = torch.optim.AdamW(
        ssl_model.parameters(), lr=SSL_LR, weight_decay=SSL_WEIGHT_DECAY,
    )

ssl_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    ssl_optimizer, T_max=SSL_EPOCHS, eta_min=1e-6
)
ssl_scaler    = GradScaler(enabled=USE_AMP and torch.cuda.is_available())

print('SSL model + losses + optimiser ready.')


Loaded glioma weights. Missing: 0  Unexpected: 0
SSL pretraining will refine these weights on meningioma anatomy.
DataParallel across 2 GPUs
SSL model parameters: 62.92 M
Fused AdamW applied
SSL model + losses + optimiser ready.


## Cell 6 — Helper: unwrap model for saving

In [6]:
def get_raw_ssl(m):
    """Strip DataParallel / torch.compile to get SwinUNETR_SSL."""
    if hasattr(m, '_orig_mod'): m = m._orig_mod
    if hasattr(m, 'module'):    m = m.module
    return m


## Cell 7 — SSL Pretraining Loop

Each epoch computes all three proxy losses:

```
L_total = W_INPAINT × L_inpaint  +  W_ROTATION × L_rotation  +  W_CONTRAST × L_contrast
        =    1.0    × L1(recon, target, mask)  
           + 1.0    × CrossEntropy(rot_logits, rot_labels)
           + 0.1    × NT-Xent(z1, z2)
```

Saves the **best encoder** (lowest total loss) every epoch.

In [ ]:
best_ssl_loss    = float('inf')
ssl_loss_history = []
log_rows         = []
run_start        = time.time()

# ── Progress widget ───────────────────────────────────────────────────────────
epoch_bar    = widgets.IntProgress(
    value=0, min=0, max=SSL_EPOCHS,
    description='SSL:', bar_style='info',
    layout=widgets.Layout(width='80%'),
)
status_label = widgets.Label(value='Starting SSL pretraining...')
display(widgets.VBox([epoch_bar, status_label]))

# ── Main SSL loop ─────────────────────────────────────────────────────────────
for epoch in range(1, SSL_EPOCHS + 1):
    ssl_model.train()
    epoch_loss_total    = 0.0
    epoch_loss_inpaint  = 0.0
    epoch_loss_rot      = 0.0
    epoch_loss_cont     = 0.0
    n_steps = 0
    t0      = time.time()

    for batch in ssl_loader:
        x_orig   = batch['x_orig'].to(DEVICE, non_blocking=True)     # (B,4,D,H,W)
        x_masked = batch['x_masked'].to(DEVICE, non_blocking=True)   # (B,4,D,H,W)
        x_aug    = batch['x_aug'].to(DEVICE, non_blocking=True)      # (B,4,D,H,W)
        x_rot    = batch['x_rot'].to(DEVICE, non_blocking=True)      # (B,4,D,H,W)
        rot_lbl  = batch['rot_label'].to(DEVICE, non_blocking=True)  # (B,)
        mask     = batch['mask'].to(DEVICE, non_blocking=True)       # (B,D,H,W) bool

        ssl_optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=USE_AMP):
            raw = get_raw_ssl(ssl_model)

            # ── Proxy Task 1: Masked Volume Inpainting ────────────────────────
            # Reconstruct the clean volume from the masked input
            # Loss only on MASKED voxels (where mask==True)
            recon        = raw.forward_inpaint(x_masked, x_orig)
            # Expand mask to (B, 4, D, H, W) for all channels
            mask_4ch     = mask.unsqueeze(1).expand_as(x_orig)
            l_inpaint    = F.l1_loss(
                recon[mask_4ch], x_orig[mask_4ch]
            )

            # ── Proxy Task 2: Rotation Prediction ────────────────────────────
            rot_logits   = raw.forward_rotation(x_rot)
            l_rotation   = F.cross_entropy(rot_logits, rot_lbl)

            # ── Proxy Task 3: Contrastive Coding ─────────────────────────────
            # z1 from original, z2 from augmented — same anatomy, different noise
            z1           = raw.forward_contrastive(x_orig)
            z2           = raw.forward_contrastive(x_aug)
            l_contrast   = contrastive_loss_fn(z1, z2)

            # ── Total loss ────────────────────────────────────────────────────
            loss = (W_INPAINT  * l_inpaint
                  + W_ROTATION * l_rotation
                  + W_CONTRAST * l_contrast)

        ssl_scaler.scale(loss).backward()
        ssl_scaler.unscale_(ssl_optimizer)
        nn.utils.clip_grad_norm_(ssl_model.parameters(), max_norm=1.0)
        ssl_scaler.step(ssl_optimizer)
        ssl_scaler.update()

        epoch_loss_total   += loss.item()
        epoch_loss_inpaint += l_inpaint.item()
        epoch_loss_rot     += l_rotation.item()
        epoch_loss_cont    += l_contrast.item()
        n_steps            += 1

    # ── Epoch averages ────────────────────────────────────────────────────────
    n = max(n_steps, 1)
    avg_total   = epoch_loss_total   / n
    avg_inpaint = epoch_loss_inpaint / n
    avg_rot     = epoch_loss_rot     / n
    avg_cont    = epoch_loss_cont    / n

    ssl_loss_history.append(avg_total)
    ssl_scheduler.step()

    elapsed = time.time() - t0
    log_rows.append({
        'epoch'       : epoch,
        'loss_total'  : round(avg_total,   5),
        'loss_inpaint': round(avg_inpaint, 5),
        'loss_rot'    : round(avg_rot,     5),
        'loss_cont'   : round(avg_cont,    5),
        'lr'          : ssl_optimizer.param_groups[0]['lr'],
        'epoch_s'     : round(elapsed, 1),
    })
    pd.DataFrame(log_rows).to_csv(SSL_LOG_PATH, index=False)

    # ── Save best ─────────────────────────────────────────────────────────────
    flag = ''
    if avg_total < best_ssl_loss:
        best_ssl_loss = avg_total
        raw = get_raw_ssl(ssl_model)
        raw.save_encoder_weights(SSL_ENCODER_PATH)
        raw.save_full_weights(SSL_FULL_PATH)
        flag = '  *** best ***'
        epoch_bar.bar_style = 'success'

    # ── Widget update ─────────────────────────────────────────────────────────
    epoch_bar.value    = epoch
    status_label.value = (
        f'Ep {epoch:03d}/{SSL_EPOCHS}  '
        f'total={avg_total:.4f}  '
        f'inpaint={avg_inpaint:.4f}  '
        f'rot={avg_rot:.4f}  '
        f'cont={avg_cont:.4f}  '
        f'lr={ssl_optimizer.param_groups[0]["lr"]:.2e}  '
        f'{elapsed:.0f}s{flag}'
    )

elapsed_total = (time.time() - run_start) / 60
status_label.value = (
    f'SSL pretraining done — {elapsed_total:.1f} min  |  '
    f'best total loss = {best_ssl_loss:.4f}'
)
epoch_bar.bar_style = 'success'


Encoder weights saved → /home/cbme/phd/bmz228464/Brats2026/GOAT/Model/model_SSL/ssl_pretrained_encoder.pth
Full backbone saved → /home/cbme/phd/bmz228464/Brats2026/GOAT/Model/model_SSL/ssl_pretrained_full.pth


## Cell 8 — SSL Loss Curves

In [ ]:
log_df = pd.read_csv(SSL_LOG_PATH)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(log_df['epoch'], log_df['loss_total'],
             lw=1.5, color='navy', label='Total')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Total SSL Loss')
axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].plot(log_df['epoch'], log_df['loss_inpaint'],
             lw=1.5, color='steelblue', label='Inpainting (L1)')
axes[1].plot(log_df['epoch'], log_df['loss_rot'],
             lw=1.5, color='darkorange', label='Rotation (CE)')
axes[1].plot(log_df['epoch'], log_df['loss_cont'],
             lw=1.5, color='seagreen', label='Contrastive')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Per-Proxy-Task Loss')
axes[1].grid(alpha=0.3); axes[1].legend()

plt.suptitle('SwinUNETR SSL Pretraining on Unlabeled Meningioma MRIs', y=1.02)
plt.tight_layout()
out_path = PRETRAIN_OUT / 'ssl_loss_curves.png'
plt.savefig(str(out_path), dpi=140, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')


## Cell 9 — Load SSL Weights into SwinUNETR for Fine-Tuning

This cell demonstrates how to load the pretrained encoder into a fresh SwinUNETR
before fine-tuning on your labeled data.

**Copy this pattern into your `brats_goat_swinunetr.ipynb` Cell 7** (model cell),
replacing the existing model construction block.

In [ ]:
from monai.networks.nets import SwinUNETR

# ── Build fresh SwinUNETR for fine-tuning ─────────────────────────────────────
finetune_model = SwinUNETR(
    in_channels       = IN_CHANNELS,
    out_channels      = OUT_CHANNELS,
    feature_size      = FEATURE_SIZE,
    use_checkpoint    = True,
    spatial_dims      = 3,
)

# ── Load SSL-pretrained ENCODER weights ───────────────────────────────────────
encoder_state = torch.load(str(SSL_ENCODER_PATH), map_location='cpu')
missing, unexpected = finetune_model.swinViT.load_state_dict(
    encoder_state, strict=True
)
print(f'SSL encoder loaded. Missing: {len(missing)}  Unexpected: {len(unexpected)}')

# ── Verify the decoder is randomly initialised (it will be trained from scratch) ──
with torch.no_grad():
    x_test = torch.randn(1, IN_CHANNELS, 96, 96, 96)
    out    = finetune_model(x_test)
    print(f'Fine-tune model forward OK: {x_test.shape} → {out.shape}')

print()
print('=' * 62)
print(' HOW TO USE IN YOUR TRAINING NOTEBOOK')
print('=' * 62)
print(f"""
In brats_goat_swinunetr.ipynb  Cell 7,
REPLACE the model construction block with:

    model = SwinUNETR(
        in_channels=4, out_channels=4,
        feature_size={FEATURE_SIZE}, use_checkpoint=True, spatial_dims=3,
    )

    # Load SSL-pretrained encoder
    encoder_state = torch.load(r'{SSL_ENCODER_PATH}', map_location='cpu')
    model.swinViT.load_state_dict(encoder_state, strict=True)
    print('SSL encoder weights loaded.')

    # Then continue with DataParallel, loss, optimizer etc.
    # Use a LOWER learning rate for the encoder, higher for decoder:
    optimizer = torch.optim.AdamW([
        {{'params': model.swinViT.parameters(), 'lr': 1e-5}},   # pretrained encoder
        {{'params': [p for n,p in model.named_parameters()
                    if 'swinViT' not in n],  'lr': 1e-4}},      # decoder (fresh)
    ], weight_decay=1e-5)
""")


## Cell 10 — Summary

In [ ]:
print('=' * 60)
print(' SSL Pretraining Complete')
print('=' * 60)
for f in sorted(PRETRAIN_OUT.rglob('*')):
    if f.is_file():
        print(f'  {f.name:<40}  {f.stat().st_size/1e6:>7.2f} MB')
print('=' * 60)
print(f'Best SSL loss     : {best_ssl_loss:.4f}')
print(f'Encoder weights   : {SSL_ENCODER_PATH}')
print(f'Full backbone     : {SSL_FULL_PATH}')
print()
print('Next steps:')
print('  1. Open brats_goat_swinunetr.ipynb')
print('  2. In Cell 7, load ssl_pretrained_encoder.pth into model.swinViT')
print('  3. Use differential LR: 1e-5 for encoder, 1e-4 for decoder')
print('  4. Train on labeled glioma data as normal')
print('  5. Expect +3-8% Dice improvement on meningioma generalisation')
